## Web3-rpc interaction

web3 library is a library we can use t interact with the blockchain . we would use this libray to pull data from the blockchain .The good thing about this library is the it has endpoints which helps in decodeing blockchaiin data efficiently.

In [ ]:
# connecting our Rpc

from web3 import Web3
RPC_URL = "https://mainnet.infura.io/v3/0179c83f73144d9eb1216769ae3f8881"


connect = Web3(Web3.HTTPProvider(RPC_URL))
print(connect.is_connected()) # Used tocheck if the connection is established.

True


In [ ]:
# Event Signature 

""" event Transfer(address indexed from, address indexed to, uint256 value); --  solidity """ 

""" Transfer (index_topic_1 address from, index_topic_2 address to, uint256 value) --Etherscan """


In [7]:
# hashing the event signature

from web3 import Web3
topic0 = Web3.keccak(text="Transfer(address,address,uint256)").hex()
print('0x' + topic0) 
topic0 = '0x'+topic0

0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4a11628f55a4df523b3ef


In [10]:
# Using the web3.py To get Transfers in the blockchain.

from_block = 26185640
to_block = 26185641


# Query the RPC node for Transfer logs within a block window
logs = connect.eth.get_logs({
    "fromBlock": from_block,
    "toBlock": to_block,
    "topics": [topic0],  # Filter ALL Transfer events across any ERC-20 contract
})

print(len(logs))

1041


In [36]:
# Manual Decoding

logs0 = logs[0]

print(logs0)
# print(logs0['transactionHash'])

transactionHash = logs0['transactionHash']
data = logs0['data']
topics = logs0['topics']
print('0x' + transactionHash.hex())
print('0x' + data.hex())

topic0 = topics[0]
topic1 = topics[1]
topic2 = topics[2]


print(topic0.hex())
print(topic1.hex())
print(topic2.hex())
# print(topics)

address_from = '0x' + topic1.hex()[-40:]

address_to = '0x' + topic2.hex()[-40:]

print(address_from)
print(address_to)


print(data)
print(int.from_bytes(data,byteorder='big'))
data = int.from_bytes(data,byteorder='big')
print(data)

AttributeDict({'removed': False, 'logIndex': 0, 'transactionIndex': 0, 'transactionHash': HexBytes('0xf28f3bd0381267d7d7db5723541f0e80ad64317994bfca996024171a5d21e44b'), 'blockHash': HexBytes('0x4ee609f7a9c0bcb4f48596858ab85fa5ce7be5436d561d2087548960d46d7c1a'), 'blockNumber': 26185640, 'blockTimestamp': '0x6ace854f', 'address': '0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48', 'data': HexBytes('0x000000000000000000000000000000000000000000000000000000012717aeae'), 'topics': [HexBytes('0xddf252ad1be2c89b69c2b068fc378daa952ba7f163c4a11628f55a4df523b3ef'), HexBytes('0x000000000000000000000000000000000004444c5dc75cb358380d2e3de08a90'), HexBytes('0x0000000000000000000000000000000aa232009084bd71a5797d089aa4edfad4')]})
0xf28f3bd0381267d7d7db5723541f0e80ad64317994bfca996024171a5d21e44b
0x000000000000000000000000000000000000000000000000000000012717aeae
ddf252ad1be2c89b69c2b068fc378daa952ba7f163c4a11628f55a4df523b3ef
000000000000000000000000000000000004444c5dc75cb358380d2e3de08a90
000000000000000000

In [ ]:
## Decoding with ABI 

contract_address = '0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48'

abi = [ {
        "anonymous": False,
        "inputs": [
            {"indexed": True, "name": "from", "type": "address"},
            {"indexed": True, "name": "to", "type": "address"},
            {"indexed": False, "name": "value", "type": "uint256"},
        ],
        "name": "Transfer",
        "type": "event",
    },]

contract = connect.eth.contract(address=contract_address,abi=abi)
events = contract.events.Transfer()
decoded = events.process_log(logs[0])
print(decoded)

address_from = decoded['from']
address_to = decoded['to']
value = decoded['value']



AttributeDict({'args': AttributeDict({'from': '0x000000000004444c5dc75cB358380D2e3dE08A90', 'to': '0x0000000aa232009084Bd71A5797d089AA4Edfad4', 'value': 4950830766}), 'event': 'Transfer', 'logIndex': 0, 'transactionIndex': 0, 'transactionHash': HexBytes('0xf28f3bd0381267d7d7db5723541f0e80ad64317994bfca996024171a5d21e44b'), 'address': '0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48', 'blockHash': HexBytes('0x4ee609f7a9c0bcb4f48596858ab85fa5ce7be5436d561d2087548960d46d7c1a'), 'blockNumber': 26185640})


In [ ]:
# Uniswap V4

## Swap (index_topic_1 bytes32 id, index_topic_2 address sender, int128 amount0, int128 amount1, uint160 sqrtPriceX96, uint128 liquidity, int24 tick, uint24 fee)

sig = 'Swap(bytes32,address,int128,int128,uint160,uint128,int24,uint24)'


topic0 = Web3.keccak(text=sig).hex()



from_block = 26185640
to_block = 26185641


# Query the RPC node for Transfer logs within a block window
logs = connect.eth.get_logs({
    "fromBlock": from_block,
    "toBlock": to_block,
    "topics": ['0x'+topic0],  # Filter ALL Transfer events across any ERC-20 contract
})

print(len(logs))


pool_address = '0x000000000004444c5dc75cB358380D2e3dE08A90'
abi = [ {
    "anonymous": False,
    "inputs": [
      {
        "indexed": True,
        "internalType": "PoolId",
        "name": "id",
        "type": "bytes32"
      },
      {
        "indexed": True,
        "internalType": "address",
        "name": "sender",
        "type": "address"
      },
      {
        "indexed": False,
        "internalType": "int128",
        "name": "amount0",
        "type": "int128"
      },
      {
        "indexed": False,
        "internalType": "int128",
        "name": "amount1",
        "type": "int128"
      },
      {
        "indexed": False,
        "internalType": "uint160",
        "name": "sqrtPriceX96",
        "type": "uint160"
      },
      {
        "indexed": False,
        "internalType": "uint128",
        "name": "liquidity",
        "type": "uint128"
      },
      {
        "indexed": False,
        "internalType": "int24",
        "name": "tick",
        "type": "int24"
      },
      {
        "indexed": False,
        "internalType": "uint24",
        "name": "fee",
        "type": "uint24"
      }
    ],
    "name": "Swap",
    "type": "event"
  },]

print(logs)
contract = connect.eth.contract(address=pool_address,abi=abi)
print(contract.address)
events = contract.events.Swap()
print(events)

decoded = events.process_log(logs[0])

print(decoded)


33
[AttributeDict({'removed': False, 'logIndex': 3, 'transactionIndex': 0, 'transactionHash': HexBytes('0xf28f3bd0381267d7d7db5723541f0e80ad64317994bfca996024171a5d21e44b'), 'blockHash': HexBytes('0x4ee609f7a9c0bcb4f48596858ab85fa5ce7be5436d561d2087548960d46d7c1a'), 'blockNumber': 26185640, 'blockTimestamp': '0x6ace854f', 'address': '0x000000000004444c5dc75cB358380D2e3dE08A90', 'data': HexBytes('0x000000000000000000000000000000000000000000000000000000012717aeaeffffffffffffffffffffffffffffffffffffffffffffffffe499b59332335ea90000000000000000000000000000000000004e034d328e7b880000184e9a914300000000000000000000000000000000000000000000000008dbc9cfb553be6c00000000000000000000000000000000000000000000000000000000000305a20000000000000000000000000000000000000000000000000000000000000000'), 'topics': [HexBytes('0x40e9cecb9f5f1f1c5b9c97dec2917b7ee92e57ba5563708daca94dd84ad7112f'), HexBytes('0xe500210c7ea6bfd9f69dce044b09ef384ec2b34832f132baec3b418208e3a657'), HexBytes('0x0000000000000000000000000000

In [4]:
logs = [1,2,3,4,5,6]

for log in logs:
    print(log)
# print(logs[5])

1
2
3
4
5
6
